In [19]:
#install requests if not already available
#--quiet flag suppresses the installation output so our notebook stays clean
!pip install requests --quiet
import pandas as pd
#numerical operation library
import numpy as np
#requests : the standard python library for making HTTP API calls
#not built in = must be installed
import requests
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
print('All libraries imported successfully!')
print(f'pandas :{pd.__version__}')
print(f'requests:{requests.__version__}')

All libraries imported successfully!
pandas :2.2.2
requests:2.32.4


In [20]:
#Extract :load raw messy sales data
raw_df = pd.read_csv('messy_sales_data.csv')
print(f"Dataset Loaded Successfully:{raw_df.shape[0]} Rows,{raw_df.shape[1]} Columns")
print(f'column names:{raw_df.columns.tolist()}')
print("\n First 5 rows")
raw_df.head(5)

Dataset Loaded Successfully:30 Rows,9 Columns
column names:['order_id', 'customer_name', 'product', 'category', 'quantity', 'unit_price', 'order_date', 'city', 'sales_rep']

 First 5 rows


,order_id,customer_name,product,category,quantity,unit_price,order_date,city,sales_rep
0,1001,Ramesh Kumar,Laptop,Electronics,2.0,45000,2024-01-05,Mumbai,Anil Sharma
1,1002,Priya Nair,NaN,Electronics,1.0,15000,2024-01-07,Delhi,Sunita Rao
2,1003,AMIT VERMA,Keyboard,Accessories,3.0,1200,2024-01-08,Bangalore,Anil Sharma
3,1004,Sunita Patel,Monitor,Electronics,NaN,22000,2024-01-10,Chennai,Ravi Kumar
4,1005,Ramesh Kumar,Laptop,Electronics,2.0,45000,2024-01-05,Mumbai,Anil Sharma


In [21]:
# Diagnose All date quality problems before fixing
print('=' * 55)
print('Data Quality Diagnosis Report')
print('=' * 55)
# isnull() return true/false for each cell
print('\n[1] Missing values per column:')
print(raw_df.isnull().sum())
#.duplicated() returns true for rows that are exact copies of a previous row
#.sum() counts how many duplicate rows exist
print(f'\n[2] Duplicate Rows:{raw_df.duplicated().sum()}')
#data types
print('\n[3] Data types:')
print(raw_df.dtypes)
#unique values in text column
print('\n[4] unique Categories:',raw_df['category'].unique())
print('[4] Sample customer names:',raw_df['customer_name'].dropna().unique()[:8])
print('[4] Sample order_date values',raw_df['order_date'].dropna().unique()[:6])


Data Quality Diagnosis Report

[1] Missing values per column:
order_id         0
customer_name    2
product          1
category         1
quantity         3
unit_price       0
order_date       0
city             0
sales_rep        0
dtype: int64

[2] Duplicate Rows:0

[3] Data types:
order_id           int64
customer_name     object
product           object
category          object
quantity         float64
unit_price         int64
order_date        object
city              object
sales_rep         object
dtype: object

[4] unique Categories: ['Electronics' 'Accessories' nan]
[4] Sample customer names: ['Ramesh Kumar' 'Priya Nair' 'AMIT VERMA' 'Sunita Patel' 'kiran mehta'
 'Deepak Singh' 'Ananya Das' 'Vikram Iyer']
[4] Sample order_date values ['2024-01-05' '2024-01-07' '2024-01-08' '2024-01-10' '07-01-2024'
 '2024-01-12']


In [22]:
#.copy() creates a completely independent copy of the dataframe
#without .copy() , df and raw_df would point to the same data
df = raw_df.copy()
print(f'Working copy created:{df.shape}')
print('raw_df is untouched - we can always resetby running df = raw_df.copy()')

Working copy created:(30, 9)
raw_df is untouched - we can always resetby running df = raw_df.copy()


In [23]:
# fix - Handle missing values
print('Before fixing nulls:',df.isnull().sum().sum(),'total missing values')
df['customer_name'].fillna('Unknown Customer',inplace=True)
median_qty = df['quantity'].median()
df['quantity'].fillna(median_qty,inplace=True)
print(f' Filled missing quantity with median:{median_qty}')
df['category'].fillna('Uncategorised',inplace=True)
print('After fixing nulls:',df.isnull().sum().sum(),'total missing values')



Before fixing nulls: 7 total missing values
 Filled missing quantity with median:2.0
After fixing nulls: 1 total missing values


In [24]:
print(f'Before deduplication:{len(df)} rows')
print(f'duplicate rows:{df.duplicated().sum()}')
print('\nDuplicate rows:')
#keep = False marks all copies of a duplicate
print(df[df.duplicated(keep=False)][['order_id','customer_name','product','order_date']].head(6))
#drop_duplicates() remove duplicates that are identical in all columns
df.drop_duplicates(inplace=True)
print(f'\nAfter deduplication:{len(df)} rows')
print(f'Rows removed:{len(raw_df) - len(df)}')


Before deduplication:30 rows
duplicate rows:0

Duplicate rows:
Empty DataFrame
Columns: [order_id, customer_name, product, order_date]
Index: []

After deduplication:30 rows
Rows removed:0


In [25]:
print('Sample dates before parsing')
print(df['order_date'].head(8).tolist())
#Some are yyyy-mm-dd some are dd-mm-yyyy
df['order_date']=pd.to_datetime(
    df['order_date'],
    dayfirst=False,
    errors='coerce' #prevent from crashing change into NAT
)
#pd.to_datetime() converts strings to proper datetime objects
#dayfirst=False:treats yyyy-mm-dd as primary format

#check for any unparseable dates(would show as NAT
nat_count=df['order_date'].isnull().sum()
print(f'\n Unparseable dates(Nat):{nat_count}')

#Extract date components-very useful for anlysis
df['year']=df['order_date'].dt.year
df['month']=df['order_date'].dt.month
df['month_name']=df['order_date'].dt.strftime('%B')
#.dt is the datetime accesro -gives access to year,month,day etc
#.strftime("%B") return the full month name:'January','Febrauary',etc.

print('\n Sample dates after parsing:')
print(df[['order_date','year','month','month_name']].head(5))

Sample dates before parsing
['2024-01-05', '2024-01-07', '2024-01-08', '2024-01-10', '2024-01-05', '07-01-2024', '2024-01-12', '2024-01-13']

 Unparseable dates(Nat):2

 Sample dates after parsing:
  order_date    year  month month_name
0 2024-01-05  2024.0    1.0    January
1 2024-01-07  2024.0    1.0    January
2 2024-01-08  2024.0    1.0    January
3 2024-01-10  2024.0    1.0    January
4 2024-01-05  2024.0    1.0    January


In [26]:
#Text standardization
print('Before standardization:',df['customer_name'].unique()[:6])
df['customer_name']=(
    df['customer_name']
    .str.strip()
    .str.title()
)
#.strip() removes spaces at edges:' Ramesh ' -> 'Ramesh'
#.title() capitalizes first letter of each letter
print('After Standardization:',df['customer_name'].unique()[:6])

#fix:Correct wrong category for keyboard rows
print(f'\nBefore :Keyboard rows with Electronics category:')
wrong_mask=(df['product']=='Keyboard') & (df['category']=='Electronics')
print(df[wrong_mask][['product','category']])
df.loc[wrong_mask,'category']='Accessories'
print('After fix: Unnique Categories:',df['category'].unique())

Before standardization: ['Ramesh Kumar' 'Priya Nair' 'AMIT VERMA' 'Sunita Patel' 'kiran mehta'
 'Deepak Singh']
After Standardization: ['Ramesh Kumar' 'Priya Nair' 'Amit Verma' 'Sunita Patel' 'Kiran Mehta'
 'Deepak Singh']

Before :Keyboard rows with Electronics category:
     product     category
23  Keyboard  Electronics
After fix: Unnique Categories: ['Electronics' 'Accessories' 'Uncategorised']


In [27]:
#Ensure numeric columns have the right types
df['quantity']=pd.to_numeric(df['quantity'],errors='coerce').astype(int)
df['unit_price']=pd.to_numeric(df['unit_price'],errors='coerce')
#pd.to_numeric() converts any column to a numeric type
#error='coerce' turns unconvertabile values to NAN instead of crashing
#.astype(int) converts float (2.0) to integer (2) for quantity

#create revenue column:quantity * unit_price
df['revenue']=df['quantity']*df['unit_price']
print('Revenue column created:')
print(df[['customer_name','product','quantity','unit_price','revenue']].head(5))
print(f'\nTotal Revenue across all orders: {df["revenue"].sum():,.0f}')


Revenue column created:
  customer_name   product  quantity  unit_price  revenue
0  Ramesh Kumar    Laptop         2       45000    90000
1    Priya Nair       NaN         1       15000    15000
2    Amit Verma  Keyboard         3        1200     3600
3  Sunita Patel   Monitor         2       22000    44000
4  Ramesh Kumar    Laptop         2       45000    90000

Total Revenue across all orders: 818,000


In [28]:
print('='*55)
print('  POST-CLEANING VALIDATION REPORT')
print('=' *55)
print(f'Original rows:{len(raw_df)}')
print(f'Cleaned rows  :{len(df)}')
print(f'Rows removed   :{len(raw_df)-(len(df))}(duplicates)')
print(f'Missing values :{df.isnull().sum().sum()}')
print(f'Duplicates     :{df.duplicated().sum()}')
print(f'Date nulls      :{df["order_date"].isnull().sum()}')
print(f'Revenue NAN    :{df["revenue"].isnull().sum()}')
print(f'Categories      :{sorted(df['category'].unique())}')
print('='*55)
print(f'Missing values:{df.isnull().sum().sum()}')
print(f'Duplicates     :{df.duplicated().sum()}')
print(f'Date nulls      :{df["order_date"].isnull().sum()}')
print(f'Revenue NAN    :{df["revenue"].isnull().sum()}')
print('=' *55)

#All checks should show clean results
all_clean=(
    df.isnull().sum().sum()==0 and

    df.duplicated().sum()==0
)
print(f'DATA IS CLEAN:{all_clean}')

  POST-CLEANING VALIDATION REPORT
Original rows:30
Cleaned rows  :30
Rows removed   :0(duplicates)
Missing values :9
Duplicates     :0
Date nulls      :2
Revenue NAN    :0
Categories      :['Accessories', 'Electronics', 'Uncategorised']
Missing values:9
Duplicates     :0
Date nulls      :2
Revenue NAN    :0
DATA IS CLEAN:False


In [29]:
product_rev = (
    df.groupby('product')['revenue']
    .sum()
    .reset_index()
    .sort_values('revenue',ascending=False)
)
print('Revenue by Product:')
print(product_rev.to_string(index=False))
category_summary = df.groupby('category').agg(
    total_revenue = ('revenue', 'sum'),
    avg_order_value = ('revenue', 'mean'),
    num_orders = ('order_id', 'count'),
    unique_products = ('product', 'nunique')
).round(2).reset_index()
print('\nCategory Summary:')
print(category_summary.to_string(index=False))


Revenue by Product:
   product  revenue
    Laptop   540000
   Monitor   154000
Headphones    28000
     Mouse    20800
  Keyboard    20400
    Webcam    20000
   USB Hub    19800

Category Summary:
     category  total_revenue  avg_order_value  num_orders  unique_products
  Accessories          81000          5785.71          14                4
  Electronics         693000         46200.00          15                3
Uncategorised          44000         44000.00           1                1


In [30]:
#to_csv save the dataframe to a csv file
df.to_csv('cleaned_sales_data.csv',index=False)
print('Cleaned data saved to cleaned_sales_data.csv')
print(f'Final dataset:{df.shape[0]} rows % {df.shape[1]} columns')
print('\n ETL pipeline for Sales Data Complete')
print('EXTRACT -> messy_sales_data.csv loaded')
print('TRANSFORM -> nulls fixed,dupes removed,dates parsed,names standardized')
print('LOAD -> cleaned_sales_data.csv saved')


Cleaned data saved to cleaned_sales_data.csv
Final dataset:30 rows % 13 columns

 ETL pipeline for Sales Data Complete
EXTRACT -> messy_sales_data.csv loaded
TRANSFORM -> nulls fixed,dupes removed,dates parsed,names standardized
LOAD -> cleaned_sales_data.csv saved


In [31]:
API_KEY='a9b13310dd57fc2e07f464bf47a776f0'
BASE_URL='http://api.openweathermap.org/data/2.5/weather'
#this is the api endpoint url
#'data/2.5/weather-the specific api route for current weather
#cities to query- 8major indian cities
CITIES=['Mumbai','Delhi','Bangalore','Chennai','Hyderabad','Kolkata','Pune','Jaipur']
#We will call the API once per city-8 major indian cities
print(f'API configured for {len(CITIES)}cities')
print(f'cities{CITIES}')

API configured for 8cities
cities['Mumbai', 'Delhi', 'Bangalore', 'Chennai', 'Hyderabad', 'Kolkata', 'Pune', 'Jaipur']


In [32]:
# ============================================================
# CELL 14 — EXTRACT: Call API for each city
# ============================================================
import requests;
def fetch_weather(city, API_KEY):
    """
    Fetch current weather data for a given city.
    Returns a dictionary with weather metrics, or None on failure.
    """
    params = {
        'q':     city,      # City name query parameter
        'appid': API_KEY,   # Authentication key
        'units': 'metric'   # Returns temperature in Celsius
    }
    # params is a dictionary — requests will encode it into the URL:
    # ?q=Mumbai&appid=KEY&units=metric

    try:
        response = requests.get(BASE_URL, params=params, timeout=10)
        # requests.get() sends an HTTP GET request to BASE_URL
        # timeout=10 — wait max 10 seconds; raise error if no response

        if response.status_code == 200:
            # status_code 200 = HTTP OK = request was successful
            data = response.json()
            # .json() parses the JSON text body into a Python dictionary

            return {
                'city':        city,
                'temperature': round(data['main']['temp'], 1),
                'feels_like':  round(data['main']['feels_like'], 1),
                'humidity':    data['main']['humidity'],
                'pressure':    data['main']['pressure'],
                'wind_speed':  data['wind']['speed'],
                'condition':   data['weather'][0]['description'].title(),
                'visibility':  data.get('visibility', 0) // 1000
                # .get('visibility', 0) — safe access: returns 0 if key missing
                # // 1000 — convert meters to kilometers (integer division)
            }
        else:
            print(f'  ERROR {response.status_code} for {city}: {response.json().get("message","unknown error")}')
            return None

    except requests.exceptions.ConnectionError:
        print(f'  CONNECTION ERROR for {city} — check internet connection')
        return None
    except requests.exceptions.Timeout:
        print(f'  TIMEOUT for {city} — API did not respond in 10 seconds')
        return None
    # try/except handles errors gracefully — one bad city doesn't crash the loop


# Call API for all cities
print('Calling Weather API...')
weather_records = []
# Empty list — we will append one dict per city

for city in CITIES:
    print(f'  Fetching: {city}...', end='')
    record = fetch_weather(city, API_KEY)
    if record:
        weather_records.append(record)
        # .append() adds the dict to our list
        print(f' {record["temperature"]}°C, {record["condition"]}')
    else:
        print(' FAILED')

print(f'\nSuccessfully fetched: {len(weather_records)}/{len(CITIES)} cities')

Calling Weather API...
  Fetching: Mumbai... 33.0°C, Haze
  Fetching: Delhi... 37.0°C, Thunderstorm
  Fetching: Bangalore... 30.6°C, Scattered Clouds
  Fetching: Chennai... 33.4°C, Scattered Clouds
  Fetching: Hyderabad... 37.2°C, Scattered Clouds
  Fetching: Kolkata... 32.0°C, Broken Clouds
  Fetching: Pune... 35.2°C, Clear Sky
  Fetching: Jaipur... 42.6°C, Haze

Successfully fetched: 8/8 cities


In [33]:
# ============================================================
# CELL 15 — Fallback data (if API key not available)
# ============================================================
# Run this cell ONLY if weather_records is empty (API not working)

if len(weather_records) == 0:
    print('Using fallback weather data (API not available)')
    weather_records = [
        {'city':'Mumbai',    'temperature':32.5,'feels_like':36.0,'humidity':78,'pressure':1009,'wind_speed':5.2,'condition':'Partly Cloudy','visibility':8},
        {'city':'Delhi',     'temperature':38.2,'feels_like':41.0,'humidity':35,'pressure':1002,'wind_speed':3.8,'condition':'Clear Sky',    'visibility':10},
        {'city':'Bangalore', 'temperature':26.1,'feels_like':27.0,'humidity':62,'pressure':1016,'wind_speed':2.5,'condition':'Overcast',     'visibility':7},
        {'city':'Chennai',   'temperature':34.8,'feels_like':39.0,'humidity':72,'pressure':1008,'wind_speed':6.1,'condition':'Hazy',         'visibility':5},
        {'city':'Hyderabad', 'temperature':35.4,'feels_like':38.5,'humidity':45,'pressure':1005,'wind_speed':4.2,'condition':'Clear Sky',    'visibility':10},
        {'city':'Kolkata',   'temperature':33.7,'feels_like':37.8,'humidity':80,'pressure':1007,'wind_speed':4.8,'condition':'Humid',        'visibility':6},
        {'city':'Pune',      'temperature':29.3,'feels_like':31.0,'humidity':55,'pressure':1014,'wind_speed':3.1,'condition':'Partly Cloudy','visibility':9},
        {'city':'Jaipur',    'temperature':40.1,'feels_like':43.0,'humidity':22,'pressure':998, 'wind_speed':5.5,'condition':'Sunny',        'visibility':12},
    ]
    print(f'Fallback data loaded for {len(weather_records)} cities')
else:
    print(f'Using live API data for {len(weather_records)} cities')

Using live API data for 8 cities


In [34]:
# ============================================================
# CELL 16 — TRANSFORM: Build DataFrame from API results
# ============================================================

weather_df = pd.DataFrame(weather_records)
# pd.DataFrame() converts a list of dictionaries into a DataFrame
# Each dictionary becomes one row
# Each dictionary key becomes a column name
# This is the standard pattern for API → DataFrame conversion

print('Weather DataFrame created:')
print(weather_df.to_string(index=False))
print(f'\nShape: {weather_df.shape}')
print(f'Missing values: {weather_df.isnull().sum().sum()}')
print(f'\nData types:')
print(weather_df.dtypes)

Weather DataFrame created:
     city  temperature  feels_like  humidity  pressure  wind_speed        condition  visibility
   Mumbai         33.0        40.0        62      1008        5.14             Haze           6
    Delhi         37.0        41.7        41       999        6.69     Thunderstorm           4
Bangalore         30.6        33.3        57      1010        2.57 Scattered Clouds           8
  Chennai         33.4        40.4        69      1005        5.14 Scattered Clouds           6
Hyderabad         37.2        39.4        34      1004        4.63 Scattered Clouds           6
  Kolkata         32.0        39.0        70      1005        2.57    Broken Clouds           3
     Pune         35.2        34.8        29      1008        6.87        Clear Sky          10
   Jaipur         42.6        42.0        17       999        6.69             Haze           5

Shape: (8, 8)
Missing values: 0

Data types:
city            object
temperature    float64
feels_like     fl

In [35]:
# ============================================================
# CELL 17 — TRANSFORM: Analysis on weather data
# ============================================================

print('=' * 50)
print('  WEATHER ANALYSIS REPORT — 8 INDIAN CITIES')
print('=' * 50)

# Hottest and coldest city
hottest = weather_df.loc[weather_df['temperature'].idxmax()]
coldest = weather_df.loc[weather_df['temperature'].idxmin()]
# .idxmax() returns the INDEX of the maximum value
# .idxmin() returns the INDEX of the minimum value
# df.loc[index] returns the full row at that index

print(f"\nHottest city : {hottest['city']} at {hottest['temperature']}°C")
print(f"Coldest city : {coldest['city']} at {coldest['temperature']}°C")
print(f"Most humid   : {weather_df.loc[weather_df['humidity'].idxmax()]['city']} "
      f"({weather_df['humidity'].max()}%)")

# Summary statistics
print(f"\nAverage temperature : {weather_df['temperature'].mean():.1f}°C")
print(f"Average humidity    : {weather_df['humidity'].mean():.1f}%")
print(f"Average wind speed  : {weather_df['wind_speed'].mean():.1f} m/s")

# Rankings
print('\nCities ranked by temperature (hottest first):')
ranked = weather_df[['city','temperature','humidity']].sort_values('temperature', ascending=False)
print(ranked.to_string(index=False))

  WEATHER ANALYSIS REPORT — 8 INDIAN CITIES

Hottest city : Jaipur at 42.6°C
Coldest city : Bangalore at 30.6°C
Most humid   : Kolkata (70%)

Average temperature : 35.1°C
Average humidity    : 47.4%
Average wind speed  : 5.0 m/s

Cities ranked by temperature (hottest first):
     city  temperature  humidity
   Jaipur         42.6        17
Hyderabad         37.2        34
    Delhi         37.0        41
     Pune         35.2        29
  Chennai         33.4        69
   Mumbai         33.0        62
  Kolkata         32.0        70
Bangalore         30.6        57


In [36]:
# ============================================================
# CELL 18 — LOAD: Save weather data to CSV
# ============================================================

weather_df.to_csv('weather_data.csv', index=False)
# Saves the cleaned, structured weather DataFrame to a CSV file
# This file can be loaded into SQLite (Day 2 skills) or used in ML (Day 5)

print('Weather data saved to: weather_data.csv')
print('\nWeather ETL Pipeline: COMPLETE')
print('  EXTRACT   → OpenWeatherMap API called for 8 cities')
print('  TRANSFORM → JSON parsed, DataFrame built, units converted')
print('  LOAD      → weather_data.csv saved')

Weather data saved to: weather_data.csv

Weather ETL Pipeline: COMPLETE
  EXTRACT   → OpenWeatherMap API called for 8 cities
  TRANSFORM → JSON parsed, DataFrame built, units converted
  LOAD      → weather_data.csv saved


1. Weather Data ETL Project Create a Python program that fetches weather data from the OpenWeatherMap API, cleans the data using Pandas, and stores the final output in a CSV file.

In [37]:
import requests
import pandas as pd

# --- EXTRACT --- #

API_KEY = '8ce86da271c6e3a978c2359e88324349' # Replace with your actual OpenWeatherMap API key
BASE_URL = 'http://api.openweathermap.org/data/2.5/weather'
CITIES = ['Mumbai', 'Delhi', 'Bangalore', 'Chennai', 'Hyderabad', 'Kolkata', 'Pune', 'Jaipur']

def fetch_weather(city, API_KEY):
    """
    Fetch current weather data for a given city.
    Returns a dictionary with weather metrics, or None on failure.
    """
    params = {
        'q':     city,
        'appid': API_KEY,
        'units': 'metric'
    }
    try:
        response = requests.get(BASE_URL, params=params, timeout=10)
        if response.status_code == 200:
            data = response.json()
            return {
                'city':        city,
                'temperature': round(data['main']['temp'], 1),
                'feels_like':  round(data['main']['feels_like'], 1),
                'humidity':    data['main']['humidity'],
                'pressure':    data['main']['pressure'],
                'wind_speed':  data['wind']['speed'],
                'condition':   data['weather'][0]['description'].title(),
                'visibility':  data.get('visibility', 0) // 1000
            }
        else:
            print(f'  ERROR {response.status_code} for {city}: {response.json().get("message","unknown error")}')
            return None

    except requests.exceptions.ConnectionError:
        print(f'  CONNECTION ERROR for {city} — check internet connection')
        return None
    except requests.exceptions.Timeout:
        print(f'  TIMEOUT for {city} — API did not respond in 10 seconds')
        return None


print('Calling Weather API...')
weather_records = []
for city in CITIES:
    print(f'  Fetching: {city}...', end='')
    record = fetch_weather(city, API_KEY)
    if record:
        weather_records.append(record)
        print(f' {record["temperature"]}°C, {record["condition"]}')
    else:
        print(' FAILED')

print(f'\nSuccessfully fetched: {len(weather_records)}/{len(CITIES)} cities')

# --- TRANSFORM --- #

# Create DataFrame from API results
weather_df = pd.DataFrame(weather_records)

print('\nWeather DataFrame created and data cleaned (structured):')
print(weather_df.head())
print(f'Shape: {weather_df.shape}')
print(f'Missing values: {weather_df.isnull().sum().sum()}')

# --- LOAD --- #

# Save the DataFrame to a CSV file
output_filename = 'weather_data_etl_output.csv'
weather_df.to_csv(output_filename, index=False)

print(f'\nWeather data saved to: {output_filename}')
print('\nWeather Data ETL Pipeline: COMPLETE')
print('  EXTRACT   → OpenWeatherMap API called')
print('  TRANSFORM → JSON parsed, DataFrame built, units converted')
print(f'  LOAD      → {output_filename} saved')


Calling Weather API...
  Fetching: Mumbai... 33.0°C, Haze
  Fetching: Delhi... 37.0°C, Thunderstorm
  Fetching: Bangalore... 30.6°C, Scattered Clouds
  Fetching: Chennai... 33.4°C, Scattered Clouds
  Fetching: Hyderabad... 37.2°C, Scattered Clouds
  Fetching: Kolkata... 32.0°C, Broken Clouds
  Fetching: Pune... 35.2°C, Clear Sky
  Fetching: Jaipur... 42.6°C, Haze

Successfully fetched: 8/8 cities

Weather DataFrame created and data cleaned (structured):
        city  temperature  feels_like  humidity  pressure  wind_speed  \
0     Mumbai         33.0        40.0        62      1008        5.14   
1      Delhi         37.0        41.7        41       999        6.69   
2  Bangalore         30.6        33.3        57      1010        2.57   
3    Chennai         33.4        40.4        69      1005        5.14   
4  Hyderabad         37.2        39.4        34      1004        4.63   

          condition  visibility  
0              Haze           6  
1      Thunderstorm           4  
2

2.Employee Salary ETL Project Load employee salary data from Excel, remove duplicate records, calculate yearly salary, and export the cleaned dataset.

In [38]:
import pandas as pd

# --- CONFIGURATION --- #
excel_file_path = 'employee_salary_data.xlsx' # Make sure this Excel file is uploaded or accessible
output_csv_file = 'cleaned_employee_salary.csv'

# --- EXTRACT --- #
print('--- Starting Employee Salary ETL Project ---')

try:
    # Attempt to load data from Excel
    df_raw = pd.read_excel(excel_file_path)
    print(f'Successfully loaded data from {excel_file_path}. Shape: {df_raw.shape}')
except FileNotFoundError:
    print(f'Warning: {excel_file_path} not found. Creating dummy data for demonstration.')
    # Create a dummy DataFrame if the Excel file is not found
    data = {
        'Employee ID': [101, 102, 103, 104, 105, 101, 106, 107, 108],
        'Name': ['Alice Smith', 'Bob Johnson', 'Charlie Brown', 'Diana Prince', 'Eve Adams', 'Alice Smith', 'Frank White', 'Grace Black', 'Harry Green'],
        'Department': ['HR', 'IT', 'Finance', 'Marketing', 'HR', 'HR', 'IT', 'Finance', 'Marketing'],
        'Monthly Salary': [5000, 6000, 7500, 5500, 5200, 5000, 6200, 7800, 5800]
    }
    df_raw = pd.DataFrame(data)
    print(f'Dummy DataFrame created. Shape: {df_raw.shape}')

df = df_raw.copy() # Create a working copy

# --- TRANSFORM --- #
print('\n--- Data Transformation ---')

# 1. Remove duplicate records
print(f'Before dropping duplicates: {len(df)} rows')
duplicates_before = df.duplicated().sum()
df.drop_duplicates(inplace=True)
print(f'Removed {duplicates_before} duplicate rows. After dropping duplicates: {len(df)} rows')

# 2. Calculate Yearly Salary
if 'Monthly Salary' in df.columns:
    df['Yearly Salary'] = df['Monthly Salary'] * 12
    print('Calculated "Yearly Salary" based on "Monthly Salary".')
elif 'salary' in df.columns:
    df['Yearly Salary'] = df['salary'] * 12
    print('Calculated "Yearly Salary" based on "salary".')
else:
    print('Could not find "Monthly Salary" or "salary" column to calculate yearly salary.')

print('\nTransformed Data (first 5 rows):')
print(df.head())
print(f'Final DataFrame shape after transformations: {df.shape}')
print(f'Missing values after transformations: {df.isnull().sum().sum()}')

# --- LOAD --- #
print('\n--- Data Loading ---')
df.to_csv(output_csv_file, index=False)
print(f'Cleaned employee salary data saved to: {output_csv_file}')

print('\n--- Employee Salary ETL Project: COMPLETE ---')
print('  EXTRACT   → Data loaded from Excel (or dummy data generated)')
print('  TRANSFORM → Duplicates removed, Yearly Salary calculated')
print(f'  LOAD      → {output_csv_file} saved')

--- Starting Employee Salary ETL Project ---
Dummy DataFrame created. Shape: (9, 4)

--- Data Transformation ---
Before dropping duplicates: 9 rows
Removed 1 duplicate rows. After dropping duplicates: 8 rows
Calculated "Yearly Salary" based on "Monthly Salary".

Transformed Data (first 5 rows):
   Employee ID           Name Department  Monthly Salary  Yearly Salary
0          101    Alice Smith         HR            5000          60000
1          102    Bob Johnson         IT            6000          72000
2          103  Charlie Brown    Finance            7500          90000
3          104   Diana Prince  Marketing            5500          66000
4          105      Eve Adams         HR            5200          62400
Final DataFrame shape after transformations: (8, 5)
Missing values after transformations: 0

--- Data Loading ---
Cleaned employee salary data saved to: cleaned_employee_salary.csv

--- Employee Salary ETL Project: COMPLETE ---
  EXTRACT   → Data loaded from Excel (or dum